<a href="https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one content item, for one client, on one date.** The warehouse fact table `fact_content_daily_performance` is a *daily panel*: each row is a single page's measured search and engagement performance for a single day — not a 90-day summary like the starter CSV.

**Grain columns:** `client_hash_id`, `content_hash_id`, `report_date`. Together these should be unique, and the query below checks exactly that.

**Time window.** I develop on `month=2026-03`, a mid-panel month. The full panel runs **2025-01-27 to 2026-06-30**, but I deliberately avoid the final month (June 2026): it is the natural outcome window for any past→future label, so developing there would place my label logic inside my own test period.

**Why this grain matters for my lane.** A reviewer decides about a *page*, not a page-day, so my modeling grain is one row per content item. The daily grain is the raw material I aggregate from — features come from a *backward* window of daily rows, and the label from a *forward* window. Keeping those two windows separate is only possible because the fact table is daily.

In [1]:
!pip install -q duckdb

import duckdb, pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN").strip()
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL       = "hf://datasets/FlyRank/internship-warehouse"
CLIENTS   = f"{REL}/dim_clients.parquet"
CONTENT   = f"{REL}/dim_content.parquet"
DEV_MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet"

# Claim: one row = one content item, one client, one date.
# If that holds, this returns zero rows.
dupes = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('{DEV_MONTH}')
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows violating the stated grain: {len(dupes)}")
print(dupes if len(dupes) else "Grain holds — no duplicate (client, content, date) combinations.")

span = con.sql(f"""
    SELECT MIN(report_date) AS first_day,
           MAX(report_date) AS last_day,
           COUNT(*)         AS rows
    FROM read_parquet('{DEV_MONTH}')
""").df()
print(span.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the stated grain: 0
Grain holds — no duplicate (client, content, date) combinations.
 first_day   last_day    rows
2026-03-01 2026-03-31 9841378


## 2. Fields: feature / label / context / excluded

For my lane I touch three tables: `dim_clients`, `dim_content`, and `fact_content_daily_performance`. Every field I plan to use goes in exactly one bucket.

**Features** — knowable *before* the moment of decision, aggregated from a backward window of daily rows:

| Field | Source | Knowable at decision time because… |
|---|---|---|
| `gsc_impressions` | fact | measured search demand already recorded in the past window |
| `gsc_clicks` | fact | past clicks, already logged before the decision |
| `gsc_avg_position` | fact | past ranking position, recorded daily |
| `ga4_sessions` / `ga4_engaged_sessions` | fact | past engagement, already measured |
| `search_volume`, `competition`, `cpc` | dim_content | keyword attributes known at publication, not outcomes |
| `word_count`, `char_count`, `backlinks` | dim_content | page attributes observable now |
| `last_optimized_date`, `content_created_date` | dim_content | dates in the past; age and staleness derive from them |
| `content_type`, `main_intent` | dim_content | static page classification |

**Label / proxy** — what I predict, and anything it is computed from. Never a feature:

- Forward-window decline in `gsc_clicks` / `gsc_impressions` — measured over the 30 days *after* the feature window. Any aggregate of the fact table inside that forward window is label material and is barred from the feature side.
- In the starter CSV the equivalent proxy was `trend_direction` / `trend_pct`; both were label-derived and never features. The warehouse version replaces that defined rule with an *observed* forward outcome.

**Context** — for joining, grouping, splitting and reading. Never learned from:

- `client_hash_id`, `content_hash_id`, `keyword_hash_id`, `url_hash_id` — salted pseudonyms. Used to join tables and to split train/test by client so the model is scored on clients it never saw. As features they would let the model memorise *which client* instead of *which page*.
- `report_date`, `month` — window construction and partition selection.
- `gsc_data_available`, `ga4_data_available` — availability filters, not signals about the page.

**Excluded** — with a reason each:

| Field | Why excluded |
|---|---|
| `is_deleted` | reflects a decision already taken about the page; a deleted page is not a review candidate |
| `provider_used`, `model_used` | records which AI system generated the content — an internal pipeline detail, not a property of how the page performs |
| `gsc_sum_position` | redundant with `gsc_avg_position` and scale-dependent on impression count; keeping both invites double-counting the same signal |
| `optimization_eligible_date` | a product-rule output — it encodes FlyRank's existing eligibility logic, so learning from it means learning the rule I am trying to beat |
| `ai_chatgpt` … `ai_other` | AI-referral columns are extremely sparse in this panel; too thin to support a feature, and treated as EDA only |

**One thing I deliberately exclude:** `optimization_eligible_date`. It is a product-decision flag, and the whole point of my lane is to test whether a learned ranking beats FlyRank's existing rules. Feeding a rule's output back in as a feature would make any "improvement" circular.

The query below confirms both: AI-referral sessions appear in only **0.056%** of March rows (5,534 of 9.8M), and `gsc_sum_position` correlates **0.604** with impressions against just **0.116** with average position — it tracks impression volume, not ranking, so it would double-count a signal `gsc_avg_position` already carries.

In [2]:
# Claim: AI-referral columns are too sparse to use as features.
ai_sparsity = con.sql(f"""
    SELECT COUNT(*) AS rows,
           SUM(CASE WHEN sessions_ai > 0 THEN 1 ELSE 0 END) AS rows_with_ai,
           ROUND(100.0 * SUM(CASE WHEN sessions_ai > 0 THEN 1 ELSE 0 END) / COUNT(*), 4) AS pct_with_ai
    FROM read_parquet('{DEV_MONTH}')
""").df()
print("AI-referral sparsity in 2026-03:")
print(ai_sparsity.to_string(index=False))

# Claim: gsc_sum_position is redundant with gsc_avg_position.
redundancy = con.sql(f"""
    SELECT ROUND(CORR(gsc_sum_position, gsc_avg_position), 3) AS corr_sum_vs_avg,
           ROUND(CORR(gsc_sum_position, gsc_impressions), 3)  AS corr_sum_vs_impressions
    FROM read_parquet('{DEV_MONTH}')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
""").df()
print("\nWhy gsc_sum_position is dropped:")
print(redundancy.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

AI-referral sparsity in 2026-03:
   rows  rows_with_ai  pct_with_ai
9841378        5534.0       0.0562


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Why gsc_sum_position is dropped:
 corr_sum_vs_avg  corr_sum_vs_impressions
           0.116                    0.604


## 3. Verify it with queries (grain, counts, missing values, windows)

**Three verified facts.** The grain check and date span are in section 1 (zero duplicate `(client, content, date)` rows; 9,841,378 rows spanning 2026-03-01 to 2026-03-31). The third fact — availability — is checked below with `IS TRUE`.

**Why availability is not the same as zero traffic.** A row where `gsc_data_available` is false means the client had no Search Console connection on that date, not that the page got no impressions. Treating those as zeros would teach the model that unconnected clients have failing pages. Filtering with `IS TRUE` keeps only rows where absence of traffic is a real measurement.

**Window split.** To build an honest past→future label inside one month, I split March: days 1–21 are the *feature* window, days 22–31 the *outcome* window. Features are computed only from the feature window; the label only from the outcome window. The two never overlap.

Measured: **36.7%** of March rows have GSC data (3,611,061 of 9,841,378), and only **4.2%** have GA4 (413,966). Both together: 364,347 rows. So unfiltered, the majority of "zero traffic" would actually be *no connection* — which is why every query filters on `gsc_data_available IS TRUE`.

**Five features, and why each is knowable at the decision moment:**

| Feature | Available when? |
|---|---|
| `impressions_21d` | summed over days 1–21, all before the decision date |
| `clicks_21d` | same past window; clicks already logged |
| `avg_position_21d` | daily position averaged over the past window; zeros excluded because `gsc_avg_position = 0` means no data, not rank one |
| `days_observed` | count of days in the 1–21 window with GSC data for this page; entirely past-window |
| `ctr_21d` | derived from two past-window columns, so it carries no future information |

**A contract decision made from the data.** I originally planned `engaged_sessions_21d` as a fifth feature, but GA4 covers only 4.2% of rows, so it was null for nearly every page. Filling those nulls with zero would encode *which clients have GA4 connected* rather than how pages perform — patterned missingness, not random. I replaced it with `days_observed`, which comes from the GSC data already filtered with `IS TRUE`.

**Frame built:** 145,091 pages with a label base rate of 26.8%.

**The deliberate leak.** The label is whether the daily click rate fell from days 1–21 to days 22–31. So `clicks_out` — total clicks in the outcome window — is label material: the label is computed from it. Adding it as a feature is the classic leak, and it is easy to do by accident when features and labels come from the same table.

I add it on purpose, score the model, then remove it and keep the honest number.


In [3]:
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_rows
    FROM read_parquet('{DEV_MONTH}')
""").df()
print("Availability in 2026-03:")
print(avail.to_string(index=False))
print(f"\nGSC coverage: {100*avail.gsc_rows[0]/avail.total_rows[0]:.1f}%")

Availability in 2026-03:
 total_rows  gsc_rows  ga4_rows  both_rows
    9841378 3611061.0  413966.0   364347.0

GSC coverage: 36.7%


In [4]:
FEAT_END  = "2026-03-21"   # features: days 1-21
OUT_START = "2026-03-22"   # outcome:  days 22-31

frame = con.sql(f"""
    WITH feat AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)            AS impressions_21d,
               SUM(gsc_clicks)                 AS clicks_21d,
               AVG(NULLIF(gsc_avg_position,0)) AS avg_position_21d,
               SUM(ga4_engaged_sessions)       AS engaged_sessions_21d,
               COUNT(*)                        AS days_observed
        FROM read_parquet('{DEV_MONTH}')
        WHERE gsc_data_available IS TRUE AND report_date <= DATE '{FEAT_END}'
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) > 0
    ),
    outcome AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS clicks_out,
               COUNT(*)        AS days_out
        FROM read_parquet('{DEV_MONTH}')
        WHERE gsc_data_available IS TRUE AND report_date >= DATE '{OUT_START}'
        GROUP BY 1, 2
    )
    SELECT f.*,
           COALESCE(o.clicks_out, 0) AS clicks_out,
           COALESCE(o.days_out, 0)   AS days_out
    FROM feat f
    LEFT JOIN outcome o USING (client_hash_id, content_hash_id)
    WHERE o.days_out > 0
""").df()

# Label: daily click rate fell from the feature window to the outcome window
frame["rate_before"] = frame.clicks_21d / frame.days_observed
frame["rate_after"]  = frame.clicks_out / frame.days_out
frame["is_declining"] = (frame.rate_after < frame.rate_before).astype(int)

# Five features, all from the feature window only
frame["ctr_21d"] = frame.clicks_21d / frame.impressions_21d

FEATURES = ["impressions_21d", "clicks_21d", "avg_position_21d", "days_observed", "ctr_21d"]
print(frame[FEATURES].isna().mean().round(4))
frame[FEATURES + ["is_declining"]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

impressions_21d     0.0000
clicks_21d          0.0000
avg_position_21d    0.0042
days_observed       0.0000
ctr_21d             0.0000
dtype: float64


,impressions_21d,clicks_21d,avg_position_21d,days_observed,ctr_21d,is_declining
0,63.0,0.0,4.281905,18,0.000000,0
1,6623.0,9.0,8.519602,21,0.001359,0
2,41.0,0.0,6.244444,18,0.000000,0
3,536.0,1.0,4.897089,21,0.001866,1
4,44.0,0.0,16.126736,17,0.000000,0


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def score(features, label="is_declining"):
    X, y, g = frame[features], frame[label], frame.client_hash_id
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42).split(X, y, g))
    m = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, random_state=42, n_jobs=-1)
    m.fit(X.iloc[tr].fillna(-1), y.iloc[tr])
    return roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te].fillna(-1))[:, 1])

honest = score(FEATURES)
print(f"Honest features only:     ROC-AUC = {honest:.3f}")

LEAKY = FEATURES + ["clicks_out"]          # clicks_out is what the label is computed from
leaked = score(LEAKY)
print(f"With leaked clicks_out:   ROC-AUC = {leaked:.3f}")
print(f"\nJump from leakage: +{leaked - honest:.3f}")

# Remove the leak. The honest number is the one that counts.
del LEAKY
print(f"\nLeak removed. Honest ROC-AUC kept: {honest:.3f}")

Honest features only:     ROC-AUC = 0.923
With leaked clicks_out:   ROC-AUC = 0.999

Jump from leakage: +0.077

Leak removed. Honest ROC-AUC kept: 0.923


In [6]:
z = frame.clicks_21d == 0
print(f"Pages with zero clicks in the feature window: {z.mean():.1%}")
print(f"  their decline rate: {frame.loc[z, 'is_declining'].mean():.1%}")
print(f"  decline rate among pages with clicks: {frame.loc[~z, 'is_declining'].mean():.1%}")

print(f"\nROC-AUC using clicks_21d alone: {score(['clicks_21d']):.3f}")
print(f"ROC-AUC on pages with clicks only: ", end="")
sub = frame[~z]
print(f"{len(sub):,} pages, base rate {sub.is_declining.mean():.1%}")

Pages with zero clicks in the feature window: 59.3%
  their decline rate: 0.0%
  decline rate among pages with clicks: 65.9%

ROC-AUC using clicks_21d alone: 0.907
ROC-AUC on pages with clicks only: 59,122 pages, base rate 65.9%


In [7]:
frame_c = frame[frame.clicks_21d > 0].copy()
print(f"Pages with clicks: {len(frame_c):,}   base rate: {frame_c.is_declining.mean():.1%}")

def score_on(df, features, label="is_declining"):
    X, y, g = df[features], df[label], df.client_hash_id
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42).split(X, y, g))
    m = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, random_state=42, n_jobs=-1)
    m.fit(X.iloc[tr].fillna(-1), y.iloc[tr])
    return roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te].fillna(-1))[:, 1])

print(f"\nHonest ROC-AUC, pages with clicks only: {score_on(frame_c, FEATURES):.3f}")
print(f"  clicks_21d alone on this subset:      {score_on(frame_c, ['clicks_21d']):.3f}")

Pages with clicks: 59,122   base rate: 65.9%

Honest ROC-AUC, pages with clicks only: 0.630
  clicks_21d alone on this subset:      0.560


**What the leak experiment showed.** Adding `clicks_out` — the column the label is computed from — pushed ROC-AUC from 0.923 to **0.999**. A near-perfect score on real-world data is a symptom, not an achievement. I removed the column.

**But 0.923 was not honest either.** The suspicious part was that removing an obvious leak still left a very high score, so I checked why. The label is `rate_after < rate_before`, and `rate_before = clicks_21d / days_observed` — both features. For any page with zero clicks in the feature window, `rate_before = 0`, and a rate cannot fall below zero, so the label can *never* fire.

Measured: **59.3%** of the 145,091 pages have zero clicks in the feature window, and their decline rate is exactly **0.0%**. `clicks_21d` alone scores **0.907**. The model was not predicting decline; it was learning "no clicks means the label cannot fire," which is true by construction.

**The honest number.** Restricting to the 59,122 pages where clicks are non-zero — where the label can genuinely go either way — gives ROC-AUC **0.630**, with `clicks_21d` alone falling to **0.560**. That is the number I keep.

**The lesson.** The leak I planted deliberately was easy to spot and easy to remove. The one that mattered came from the label *definition*, not from a misplaced column, and it survived removing the obvious leak. A score that looks too good is worth interrogating even after the known problem is fixed.

## 4. Data limits

**Named limitation: the label is undefined for most of the panel.** 59.3% of pages have zero clicks in the feature window, so a "did the click rate fall" label cannot fire for them. My honest scope is the 59,122 pages with non-zero clicks — 40.7% of the March frame. For the rest, this framing has nothing to say, and a refresh queue built from it silently ignores them.

**Availability is not zero traffic.** Only 36.7% of March rows have GSC data and 4.2% have GA4. Absence of traffic usually means the client had no connection, not that the page failed — which is why every query filters with `IS TRUE`.

**Unbalanced history, measured.** The 104 clients enter the panel between **2025-01-27 and 2026-06-02** — a spread of nearly 17 months. A page with no early data may be new to *tracking*, not new to the web. Only 54 clients have GA4 at all, and 26 have `no_search_or_analytics_access`, so more than a quarter of clients can contribute no search or engagement signal whatsoever. A further 10 are `source_only_missing_client_dimension`, meaning they appear in the fact table without a full client record.

**One month is not a season.** Everything here is measured on March 2026 alone. Seasonal effects, algorithm updates and client campaigns are

In [8]:
starts = con.sql(f"""
    SELECT COUNT(*) AS clients,
           MIN(gsc_data_start) AS earliest_gsc,
           MAX(gsc_data_start) AS latest_gsc,
           SUM(CASE WHEN has_ga4_access IS TRUE THEN 1 ELSE 0 END) AS with_ga4
    FROM read_parquet('{CLIENTS}')
""").df()
print(starts.to_string(index=False))

print("\nAccess profiles:")
print(con.sql(f"""
    SELECT access_profile, COUNT(*) AS n
    FROM read_parquet('{CLIENTS}')
    GROUP BY 1 ORDER BY 2 DESC
""").df().to_string(index=False))

 clients earliest_gsc latest_gsc  with_ga4
     104   2025-01-27 2026-06-02      54.0

Access profiles:
                      access_profile  n
                         gsc_and_ga4 53
       no_search_or_analytics_access 26
                            gsc_only 14
source_only_missing_client_dimension 10
                            ga4_only  1


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.